In [1]:
from mlx_lm import load, generate

model, tokenizer = load("mlx-community/gpt-oss-120b-MXFP4-Q4")

/Users/yurt/Workshop/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

In [2]:
import xml.etree.ElementTree as ET

test_tree = ET.parse('./archehr-qa.xml')
test_root = test_tree.getroot()

test_patient = [i.text.strip() for i in test_root.iter('patient_narrative')]
test_ids = [i.attrib['id'] for i in test_root.iter('case')]

dev_tree = ET.parse('./data/dev/archehr-qa.xml')
dev_root = dev_tree.getroot()

dev_patient = [i.text.strip() for i in dev_root.iter('patient_narrative')]
dev_clinician = [i.text.strip() for i in dev_root.iter('clinician_question')]

print(len(test_patient), len(test_ids), len(dev_patient), len(dev_clinician))

47 47 20 20


In [3]:
import numpy as np
import torch
from evaluate import load
from enum import Enum

class BertScorer:
    def __init__(self, device):
        self.device = device
        self.bertscore = load("bertscore")

    def compute_scores(self, references, predictions):
        scores = self.bertscore.compute(
            references=references,
            predictions=predictions,
            model_type="distilbert-base-uncased",
            device=self.device,
            num_layers=4,
            batch_size=8,
            nthreads=4,
            all_layers=False,
            idf=False,
            lang="en",
            rescale_with_baseline=True,
            baseline_path=None,
        )
        return scores["f1"]

    def compute_overall_score(self, references, predictions):
        scores = self.compute_scores(references, predictions)
        return np.mean(scores)


class RougeType(Enum):
    ROUGE1 = "rouge1"
    ROUGE2 = "rouge2"
    ROUGEL = "rougeL"
    ROUGELSUM = "rougeLsum"


class RougeScorer:
    def __init__(self, rouges=["rouge1", "rouge2", "rougeL", "rougeLsum"]):
        self.rouge = load("rouge")
        self.rouge_types = [RougeType(rt) for rt in rouges]

    def compute_scores(self, references, predictions):
        scores = {rt.value: [] for rt in self.rouge_types}
        for r, p in zip(references, predictions):
            rouge_scores = self.rouge.compute(references=[r], predictions=[p])
            for rouge_type, rt_scores in scores.items():
                rt_scores.append(rouge_scores[rouge_type])
        return scores

    def compute_overall_score(self, references, predictions):
        scores = self.compute_scores(references, predictions)
        return {key: np.mean(value) for key, value in scores.items()}

device = "mps"

rouge_scorer = RougeScorer()
bert_scorer = BertScorer(device)

In [4]:
# prepare the model input
SYSTEM_PROMPT = f"""You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Input:\n{question}\n\nOutput:"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True
    )
    
    text = generate(model, tokenizer, prompt=prompt)

    return text

In [5]:
# prepare the model input
CLEANING_PROMPT = f"""Your task to transform a free-text, patient-authored question into a clear and concise clinician-interpreted question that reflects how a clinician would query a smart electronic health record (EHR) system when preparing a response to the patient.
Your outputs should be very brief only focusing on the key question in less than 15 words.

Goal:
    Generate a single, well-formed clinician question that captures the core clinical information need implied by the patient’s narrative, phrased as a query to an intelligent EHR system.
Input:
    Patient‑authored question (Patient Question)
Expected output:
    Clinician‑interpreted question that captures the main medical concern(s) (≤ 15 words).

Examples:
    Patient Question: {dev_patient[0]}
    Clinician-interpreted Question: {dev_clinician[0]}

    Patient Question: {dev_patient[1]}
    Clinician-interpreted Question: {dev_clinician[1]}

    Patient Question: {dev_patient[2]}
    Clinician-interpreted Question: {dev_clinician[2]}
"""

def get_answer_2(question):
    messages = [
        {"role": "system", "content": CLEANING_PROMPT},
        {"role": "user", "content": f"Patient Question:{question}\nClinician-interpreted Question:"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True
    )
    
    text = generate(model, tokenizer, prompt=prompt)

    return text

In [6]:
# prepare the model input
SYSTEM_PROMPT_2 = f"""You are converting long, messy questions into their single core question,
rewritten from a third-person clinical perspective.

Task:
- Identify what the person is really asking.
- Rewrite it as a short, clear question.
- Rewrite the question in THIRD PERSON, as a doctor or clinician would ask.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}"""

def get_answer_3(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_2},
        {"role": "user", "content": f"Input:\n{question}\n\nOutput:"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True
    )
    
    text = generate(model, tokenizer, prompt=prompt)

    return text

In [4]:
# prepare the model input
SYSTEM_PROMPT_3 = f"""You rewrite long, messy questions into ONE short core question, phrased in third person from a clinician’s perspective.

Task:
- Identify the single core question being asked.
- Rewrite it as one clear question in THIRD PERSON.

Hard Rules (must follow):
- Output exactly ONE sentence
- Maximum 15 words
- Use third-person only (e.g., the patient, he, she, they)
- Do NOT explain, summarize, or add context
- Do NOT add new information
- Output ONLY the rewritten question

Question Style:
- Use simple forms like "What", "Why", "Did", "Is", "How"

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}

Input:
{dev_patient[3]}
Output:
{dev_clinician[3]}

Input:
{dev_patient[4]}
Output:
{dev_clinician[4]}"""

def get_answer_4(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_3},
        {"role": "user", "content": f"Input:\n{question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )
    
    text = generate(model, tokenizer, prompt=prompt, max_tokens=512)

    return text

In [4]:
# prepare the model input
SYSTEM_PROMPT_3 = f"""You rewrite long, messy questions into ONE short core question, phrased in third person from a clinician’s perspective.

Task:
- Identify the single core question being asked.
- Rewrite it as one clear question in THIRD PERSON.

Hard Rules (must follow):
- Output exactly ONE sentence
- Maximum 15 words
- Use third-person only (e.g., the patient, he, she, they)
- Do NOT explain, summarize, or add context
- Do NOT add new information
- Output ONLY the rewritten question

Question Style:
- Use simple forms like "What", "Why", "Did", "Is", "How"

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}

Input:
{dev_patient[3]}
Output:
{dev_clinician[3]}

Input:
{dev_patient[4]}
Output:
{dev_clinician[4]}"""

def get_answer_4_doubleq(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_3},
        {"role": "user", "content": f"Input:\n{question}\nInput:\n{question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )
    
    text = generate(model, tokenizer, prompt=prompt, max_tokens=512)

    return text

In [4]:
SYSTEM_PROMPT = f"""You are a clinical language understanding system assisting with patient–provider communication.

## Task
Given a free-text, patient-authored question that may be long, verbose, emotional, or poorly structured, generate **one clear and concise clinician-interpreted question**. The output should reflect how a clinician would query an intelligent electronic health record (EHR) system when preparing a response to the patient.

## Objective
Identify the **core clinical information need** implied by the patient’s narrative and restate it as a focused medical question, using appropriate clinical terminology and perspective.

## Input
- A single patient-authored question written in free text (“Patient Question”)

## Output
- A single clinician-interpreted question that:
  - Captures the **primary medical concern(s)** raised by the patient
  - Is phrased as a **query suitable for an intelligent EHR system**
  - Reflects **clinical reasoning**, not patient confusion or emotion
  - Is **15 words or fewer**

## Guidelines
- Produce **one question only** (no lists, no explanations, no commentary).
- Be **specific and clinically grounded**; infer context when reasonable.
- Do **not** simply restate the patient’s wording verbatim.
- Avoid overly generic or textbook questions that are not tied to the case details.
- Exclude emotional language, speculation, or non-clinical details unless clinically relevant.
- Do not add new facts that are not implied by the patient narrative.
- Assume the clinician is asking *why*, *whether*, or *what indication* prompted an action, finding, or management decision.
- Prioritize **brevity and precision**; remove unnecessary modifiers.

## Quality Criteria
A good output:
- Sounds like a question a clinician would actually ask when reviewing the chart
- Is narrower and clearer than the patient’s original question
- Focuses on *indications, rationale, or clinical reasoning*
- Meets the **≤ 15 words** constraint

A poor output:
- Is generic (e.g., asking about general disease management)
- Ignores key context from the patient’s description
- Produces multiple questions or a fragmented query
- Exceeds the word limit

## Output Format
- Plain text
- One sentence
- Question form
- **15 words or fewer**

## Few-Shot Examples

**Patient Question:**  
{dev_patient[0]}  

**Clinician-Interpreted Question:**  
{dev_clinician[0]}

**Patient Question:**  
{dev_patient[1]}  

**Clinician-Interpreted Question:**  
{dev_clinician[1]}

**Patient Question:**  
{dev_patient[2]}  

**Clinician-Interpreted Question:**  
{dev_clinician[2]}

**Patient Question:**  
{dev_patient[3]}  

**Clinician-Interpreted Question:**  
{dev_clinician[3]}

**Patient Question:**  
{dev_patient[4]}  

**Clinician-Interpreted Question:**  
{dev_clinician[4]}"""

def get_answer_5(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question::\n{question}\n\nClinician-Interpreted Question:"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )
    
    text = generate(model, tokenizer, prompt=prompt, max_tokens=512)

    return text

In [12]:
SYSTEM_PROMPT = f"""You rewrite long, messy patient-authored questions into ONE short core question, phrased in third person from a clinician’s perspective.

Task:
Given a free-text patient question, identify the single underlying information need and rewrite it as one clear, simple clinician-style question. The question should reflect how a clinician would briefly query an intelligent EHR system, without unnecessary clinical detail.

Input:
A patient-authored question written in free text.

Output:
One rewritten clinician-interpreted question that captures the primary intent, written in third person, 15 words or fewer.

Hard Rules (must follow):

* Output exactly ONE sentence
* Maximum 15 words
* Use third person only (e.g., the patient, he, she, they)
* Do NOT explain, summarize, or add context
* Do NOT add new information
* Do NOT include multiple questions
* Output ONLY the rewritten question

Style Constraints:

* Prefer simple question forms: What, Why, Did, Is, How
* Keep language plain and minimal
* Avoid detailed medical jargon unless required to preserve meaning
* Do not include emotional or narrative elements

Quality Guidance:
A good output is a direct intent query, not a summary, and is shorter and simpler than the original question. A bad output adds clinical reasoning, unnecessary specificity, or explanation.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}

Input:
{dev_patient[3]}
Output:
{dev_clinician[3]}

Input:
{dev_patient[4]}
Output:
{dev_clinician[4]}
"""

def get_answer_6(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Input:\n{question}\n\nOutput:"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )
    
    text = generate(model, tokenizer, prompt=prompt, max_tokens=512)

    return text

In [7]:
from tqdm.notebook import tqdm

dev_answers = []
for q in tqdm(dev_patient):
    a = get_answer(q)
    dev_answers.append(a)

  0%|          | 0/20 [00:00<?, ?it/s]

In [8]:
dev_answers_2 = []
for q in tqdm(dev_patient):
    a = get_answer_2(q)
    dev_answers_2.append(a)

  0%|          | 0/20 [00:00<?, ?it/s]

In [9]:
from tqdm.notebook import tqdm

dev_answers_3 = []
for q in tqdm(dev_patient):
    a = get_answer_3(q)
    dev_answers_3.append(a)

  0%|          | 0/20 [00:00<?, ?it/s]

In [6]:
from tqdm.notebook import tqdm

dev_answers_4 = []
for q in tqdm(dev_patient):
    a = get_answer_4(q)
    dev_answers_4.append(a)

  0%|          | 0/20 [00:00<?, ?it/s]

In [5]:
from tqdm.notebook import tqdm

dev_answers_4_dq = []
for q in tqdm(dev_patient):
    a = get_answer_4_doubleq(q)
    dev_answers_4_dq.append(a)

  0%|          | 0/20 [00:00<?, ?it/s]

In [5]:
from tqdm.notebook import tqdm

dev_answers_5 = []
for q in tqdm(dev_patient):
    a = get_answer_5(q)
    dev_answers_5.append(a)

  0%|          | 0/20 [00:00<?, ?it/s]

In [14]:
from tqdm.notebook import tqdm

dev_answers_6 = []
for q in tqdm(dev_patient):
    a = get_answer_6(q)
    dev_answers_6.append(a)

  0%|          | 0/20 [00:00<?, ?it/s]

In [10]:
[i.split('<|channel|>final<|message|>')[-1] for i in dev_answers]

['Why was ERCP recommended over medication for treating the CBD sludge?',
 'Why was he given lasix and his oxygen flow rate was reduced?',
 "How should I manage my son's persistent irritation and headache after his brain bleed?",
 'Was the cardiac catheterization necessary in my case?',
 'What could be causing my persistent left upper chest pain after the overdose?',
 'Did the doctors fail to diagnose his lung infection in a timely manner?',
 'What are the guidelines for stopping Coumadin in a patient with prior leg vein surgery?',
 'Does a toxin that entered my bloodstream stay in my body permanently?',
 'What treatment is provided for my postpartum jaundice and bleeding, and will it cause future problems?',
 'Will he be able to breathe independently and survive long‑term after his asthma‑induced heart attack?',
 'Is a setback in symptoms during viral meningitis recovery normal or a sign she should return to the hospital?',
 'What could be causing my mom’s severe upper abdominal pain 

In [11]:
[i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_2]

['Why was ERCP chosen instead of medication for treating CBD sludge?',
 'Why was high‑dose Lasix given and oxygen flow reduced in his edema management?',
 'What is the appropriate management for persistent headache and irritability after mild traumatic brain bleed?',
 'Was cardiac catheterization necessary given normal enzymes and reduced ejection fraction?',
 'What is causing his persistent left upper quadrant chest pain after overdose and prolonged QT?',
 'Was the lung infection diagnosis and treatment delayed, contributing to his sudden cardiac arrest?',
 'What are recommendations for anticoagulation after stopping Coumadin in prior vein bypass patient?',
 'Does residual toxin remain in my body causing ongoing cognitive and writing difficulties?',
 'What treatment is indicated for her postpartum jaundice and bleeding, and what are future risks?',
 'What is his long‑term breathing and survival prognosis after cardiac arrest and possible brain injury?',
 "Is today's symptom worsening 

In [12]:
[i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_3]

['Why was ERCP recommended to him over continuing a medication-based treatment?',
 'Why was he given lasix and his oxygen flow rate was reduced?',
 'What is the expected course of recovery for him?',
 'Why was cardiac catheterization considered necessary for him?',
 'What could be causing his persistent left upper‑quadrant chest pain after the overdose and prolonged QT episode?',
 'Did the physicians fail to diagnose his lung infection in a timely manner?',
 'What are the recommended instructions for managing her prior vein surgery after Coumadin was stopped?',
 'Does the toxin remain in his system?',
 'What treatment is being provided for her postpartum jaundice and bleeding, and will she have any long‑term complications?',
 'Will he be able to maintain independent breathing and survive long‑term despite possible brain injury?',
 'Is the patient’s worsening symptoms today a normal part of viral meningitis recovery or a sign of relapse that warrants readmission?',
 'What is the likely 

In [8]:
[i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_4]

['Why was ERCP recommended to the patient instead of medication?',
 'Why was he given multiple Lasix doses and his oxygen flow reduced?',
 'What should be done for his persistent headache and irritability after discharge?',
 'Why was cardiac catheterization performed on the patient despite normal enzymes and low ejection fraction?',
 'Is the left upper quadrant pain related to his overdose or another cause?',
 'Did the doctor fail to diagnose his lung infection promptly?',
 'What are the guidelines for stopping Coumadin in her after her vein surgery?',
 'Does the toxin remain in his system and cause his dysgraphia?',
 'What treatment is she receiving for postpartum jaundice and bleeding, and are complications expected?',
 'Will he be able to breathe independently and survive long term?',
 'Is her worsening today a normal part of recovery or a sign of relapse?',
 'What could be causing her abdominal pain radiating to chest and back?',
 'What can the patient do to relieve pain and improv

In [6]:
[i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_4_dq]

['Why was ERCP recommended to him instead of medication for his CBD sludge?',
 'Why was he given multiple Lasix doses and his oxygen flow rate reduced?',
 'What should be done for his son’s persistent irritation and headache after discharge?',
 'Why was cardiac catheterization recommended for the patient?',
 'Is his left upper quadrant pain due to the overdose or another cause?',
 'Did the doctors fail to diagnose his lung infection promptly?',
 'What are the guidelines for stopping Coumadin in her after the subarachnoid hemorrhage?',
 'Does the toxin remain in his system and cause his writing difficulties?',
 'What treatment is she receiving for her post‑delivery jaundice and bleeding?',
 'Will he be able to breathe independently long‑term and survive after his brain injury?',
 'Is the patient’s worsening today a normal part of recovery or a sign of relapse?',
 'What could be causing her abdominal pain radiating to her chest and back?',
 'What can the patient do to alleviate pain and 

In [6]:
[i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_5]

["Why was ERCP indicated instead of medical therapy for the patient's CBD sludge?",
 'Why was high‑dose IV Lasix administered and oxygen flow reduced in his management?',
 'What is the expected course and management for post‑traumatic headache and irritability after intracranial hemorrhage?',
 'Why was cardiac catheterization indicated despite normal enzymes and reduced ejection fraction?',
 'Is his left upper quadrant chest pain due to the overdose or a cardiac complication?',
 'Why was the bacterial/fungal pneumonia not identified earlier in his ICU course?',
 'What are the guidelines for anticoagulation after lower extremity venous bypass when Coumadin is discontinued?',
 'Could residual toxins from the liver failure explain his persistent dysgraphia and neurocognitive symptoms?',
 'Why does she have postpartum jaundice and bleeding, and what is the appropriate management?',
 'What is the prognosis for independent breathing after cardiac arrest with possible brain injury?',
 'Is the

In [15]:
[i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_6]

['Why was ERCP recommended for him rather than medication to clear the CBD sludge?',
 'Why was he given high-dose Lasix and his oxygen flow reduced?',
 'What management is recommended for his persistent headache and irritability after brain bleed?',
 'Why was cardiac catheterization recommended for the patient?',
 'Is his pain due to the overdose or another cause?',
 'Did the clinicians diagnose his lung infection promptly?',
 'What are the instructions for stopping Coumadin in her after vein surgery?',
 'Does the toxin remain in his bloodstream?',
 'What treatment is she receiving for her jaundice and bleeding, and are complications expected?',
 'Will he be able to breathe independently long term?',
 'Is her worsening today normal recovery or indicating relapse requiring hospital return?',
 'What could be causing her epigastric pain radiating to chest and back?',
 'What can be done now to relieve his severe pain and immobility?',
 'Could the patient’s left lower abdominal pain be caus

In [14]:
bert_score = bert_scorer.compute_overall_score(dev_clinician[3:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers[3:]])
rouge_score = rouge_scorer.compute_overall_score(dev_clinician[3:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers[3:]])

print(bert_score, rouge_score)

bert_score = bert_scorer.compute_overall_score(dev_clinician[3:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_2[3:]])
rouge_score = rouge_scorer.compute_overall_score(dev_clinician[3:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_2[3:]])

print(bert_score, rouge_score)

bert_score = bert_scorer.compute_overall_score(dev_clinician[3:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_3[3:]])
rouge_score = rouge_scorer.compute_overall_score(dev_clinician[3:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_3[3:]])

print(bert_score, rouge_score)

0.4025623693185694 {'rouge1': np.float64(0.2488397146991843), 'rouge2': np.float64(0.044721083879472094), 'rougeL': np.float64(0.21390507417630855), 'rougeLsum': np.float64(0.21390507417630855)}
0.33552248688305125 {'rouge1': np.float64(0.18434879294525885), 'rouge2': np.float64(0.02324521916456053), 'rougeL': np.float64(0.1668417901441384), 'rougeLsum': np.float64(0.1668417901441384)}
0.4035785145619336 {'rouge1': np.float64(0.26268584555609775), 'rouge2': np.float64(0.06360932096226213), 'rougeL': np.float64(0.23307815198286372), 'rougeLsum': np.float64(0.23307815198286372)}


In [10]:
bert_score = bert_scorer.compute_overall_score(dev_clinician[5:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_4[5:]])
rouge_score = rouge_scorer.compute_overall_score(dev_clinician[5:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_4[5:]])

print(bert_score, rouge_score)

0.4114993214607239 {'rouge1': np.float64(0.2510293552681781), 'rouge2': np.float64(0.05183152689846402), 'rougeL': np.float64(0.2350922059976955), 'rougeLsum': np.float64(0.2350922059976955)}


In [7]:
bert_score = bert_scorer.compute_overall_score(dev_clinician[5:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_4_dq[5:]])
rouge_score = rouge_scorer.compute_overall_score(dev_clinician[5:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_4_dq[5:]])

print(bert_score, rouge_score)

0.4379125436147054 {'rouge1': np.float64(0.2586021923602472), 'rouge2': np.float64(0.05910425417119129), 'rougeL': np.float64(0.24237644280116427), 'rougeLsum': np.float64(0.24237644280116427)}


In [16]:
bert_score = bert_scorer.compute_overall_score(dev_clinician[5:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_6[5:]])
rouge_score = rouge_scorer.compute_overall_score(dev_clinician[5:], [i.split('<|channel|>final<|message|>')[-1] for i in dev_answers_6[5:]])

print(bert_score, rouge_score)

0.4007989664872487 {'rouge1': np.float64(0.2452270520749151), 'rouge2': np.float64(0.03612535612535613), 'rougeL': np.float64(0.21954445977240694), 'rougeLsum': np.float64(0.21954445977240694)}


In [7]:
from tqdm.notebook import tqdm

test_answers_4 = []
for q in tqdm(test_patient):
    a = get_answer_4(q)
    test_answers_4.append(a)

  0%|          | 0/47 [00:00<?, ?it/s]

In [8]:
test_answers_4_dq = []
for q in tqdm(test_patient):
    a = get_answer_4_doubleq(q)
    test_answers_4_dq.append(a)

  0%|          | 0/47 [00:00<?, ?it/s]

In [9]:
# submission_single = [{'case_id': i, 'prediction': j} for i, j in zip(test_ids, [i.split('<|channel|>final<|message|>')[-1] for i in test_answers_4])]
submission_double = [{'case_id': i, 'prediction': j} for i, j in zip(test_ids, [i.split('<|channel|>final<|message|>')[-1] for i in test_answers_4_dq])]

In [11]:
import json

# with open("./single/submission.json", "w", encoding="utf-8") as f:
#     json.dump(submission_single, f, indent=4)
with open("./double/submission.json", "w", encoding="utf-8") as f:
    json.dump(submission_double, f, indent=4)

In [13]:
[i.split('<|channel|>final<|message|>')[-1] for i in test_answers_4]

['Is it advisable for him to travel to the Maldives after his recovery?',
 'What does the fluid in his lungs and elevated troponin indicate after his heart attack?',
 'What treatment is recommended for the patient’s pneumocystis carinii pneumonia?',
 'Could the enlarged spleen and kidney indicate cancer in him?',
 'Could the breathing tube procedure be due to his diagnosis?',
 'What is the likely cause of his persistent cough and shortness of breath?',
 'What could the shadow on his brain represent?',
 'Will the patient regain her memory?',
 'What is causing his weakness, poor appetite, and rapid weight loss?',
 'Should the patient be concerned that his leg will not heal properly?',
 'What could be causing his dysuria after stent placement?',
 'How can the bleeding be stopped for his father?',
 'How should the patient safely taper his antihypertensive medications?',
 'How did the infection develop in her wound after her hip replacement?',
 'What further evaluation or treatment is recom

In [10]:
[i.split('<|channel|>final<|message|>')[-1] for i in test_answers_4_dq]

['Is it safe for him to travel to the Maldives during his recovery?',
 'What does his lung fluid and elevated troponin indicate after his heart attack?',
 'What is pneumocystis carinii and how should it be treated for the patient?',
 'Could the enlarged spleen and kidney indicate cancer in his brother?',
 'Is the breathing tube procedure required because of his pneumonia?',
 'What could be causing his persistent cough and shortness of breath despite extensive testing?',
 'What could the shadow on his brain indicate?',
 'Is there a chance of her regaining her memory?',
 'What could be causing his weakness, poor appetite, and rapid weight loss?',
 'Should the patient’s broken leg be expected to heal properly?',
 'What is causing his pain at the end of urination after stent placement?',
 'How can the bleeding in his palate be stopped and evaluated for cancer?',
 'What is the safest way to taper his antihypertensive medications?',
 'How did the infection develop in her postoperative wound?